# Social graphs: quickstart

The fastest path to one real run: a node-level hybrid GNN (α = 0.5, Performer attention) on a small set of
twitch-gamers subgraphs for 20 epochs (`configs/social/quickstart.yaml`, about 15 minutes on a T4).
**Use a GPU runtime** (Runtime > Change runtime type > T4 GPU); a CPU runtime works but is much slower.

The dataset is downloaded once into Drive and reused by every later session. The run folder lives on
Drive, so after a disconnect, rerun every cell: training resumes from its last checkpoint, and a finished
run is skipped.

## 1. Settings (edit)

`BRANCH` defaults to `main`; set it to another branch to run that code instead.

In [ ]:
# EDIT: settings for this notebook
BRANCH = 'social-pipeline'          # git branch to run; change it to test a feature branch
USE_TOKEN = False        # True if the repo is private (needs the GITHUB_TOKEN Colab secret, see step 3)
DRIVE = '/content/drive/MyDrive/Thesis/gnn_mech'
DATA_ROOT = f'{DRIVE}/data'    # downloads and processed caches (kept on Drive, reused by every session)
OUT_DIR = f'{DRIVE}/runs'      # run folders (kept on Drive so runs resume after a disconnect)
LOCAL_DATA = '/content/data'   # fast local copy of the dataset used during training
DATASET = 'twitch-gamers'
CONFIG = 'configs/social/quickstart.yaml'

## 2. Check the GPU (Runtime > Change runtime type > T4 GPU)

In [ ]:
!nvidia-smi

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Get the code

Clones the repo at `BRANCH`, or pulls if it is already there. If the repo is private, create a GitHub
personal access token (read access to the repo is enough), store it as a Colab secret named
`GITHUB_TOKEN` (key icon in the left sidebar) and set `USE_TOKEN = True` in step 1. Never paste the
token into the notebook itself.

In [ ]:
import os, subprocess

REPO_DIR = '/content/message-passing-in-gnns'
url = 'https://github.com/djdhawal/message-passing-in-gnns.git'
if USE_TOKEN:
    from google.colab import userdata
    url = f"https://{userdata.get('GITHUB_TOKEN')}@github.com/djdhawal/message-passing-in-gnns.git"

if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, url, REPO_DIR], check=True)
%cd {REPO_DIR}
!git log -1 --oneline

## 5. Install

In [ ]:
!pip install -q torch_geometric
!pip install -q -e .

## 6. Folders on Drive

In [ ]:
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)
print('data on Drive:', DATA_ROOT)
print('runs on Drive:', OUT_DIR)

## 7. Data helpers

`find_source_dir` finds the dataset's folder under `<root>/social/`. `sync` copies only files that are
missing or newer (rsync). Nothing to edit here.

In [ ]:
import shutil
from pathlib import Path


def _source_candidates(name):
    # Folder names under <root>/social/ that may hold `name`; gnn_mech.social.sources decides when it can.
    cands = []
    try:
        from gnn_mech.social import sources
        for attr in ('source_of', 'source_name', 'source_for'):
            fn = getattr(sources, attr, None)
            if callable(fn):
                try:
                    cands.append(str(fn(name)))
                except Exception:
                    pass
        for attr in ('SOURCES', 'SOURCE_OF', 'DATASET_SOURCE', 'DATASETS'):
            table = getattr(sources, attr, None)
            if isinstance(table, dict) and name in table:
                v = table[name]
                if not isinstance(v, str):
                    v = v.get('source') if isinstance(v, dict) else getattr(v, 'source', None)
                if v:
                    cands.append(str(v))
    except Exception:
        pass
    cands.append(name)
    if name.startswith('twitch-') and name != 'twitch-gamers':
        cands += ['musae', 'twitch', 'twitch-musae', 'musae-twitch']
    if name.startswith('higgs-'):
        cands += ['higgs', 'higgs-twitter']
    if name.startswith('reddit-'):
        cands += ['reddit', 'reddit-hyperlinks', 'soc-redditHyperlinks']
    return list(dict.fromkeys(cands))


def find_source_dir(name, root, fetched=None):
    # Name of the folder <root>/social/<source>/ that holds `name`, or None if it does not exist yet.
    social = Path(root) / 'social'
    if isinstance(fetched, (str, os.PathLike)):
        try:
            rel = Path(fetched).resolve().relative_to(social.resolve())
            if rel.parts:
                return rel.parts[0]
        except Exception:
            pass
    for c in _source_candidates(name):
        if (social / c).is_dir():
            return c
    return None


def sync(src, dst):
    # Copy files that are missing or newer in src into dst; never deletes anything.
    os.makedirs(dst, exist_ok=True)
    if shutil.which('rsync'):
        subprocess.run(['rsync', '-a', '--update', f'{src}/', f'{dst}/'], check=True)
    else:
        shutil.copytree(src, dst, dirs_exist_ok=True)

## 8. Download the dataset into Drive (once)

`gnn_mech.social.sources.fetch` downloads and parses the raw files into `DATA_ROOT/social/<source>/` and
skips files that already exist, so later sessions reuse them. If it fails, the run downloads the data
itself on first use.

In [ ]:
fetched = None
try:
    from gnn_mech.social import sources
    fetched = sources.fetch(DATASET, root=DATA_ROOT)
except Exception as e:
    print(f'[warn] fetch({DATASET!r}) failed: {e!r}')
SOURCE = find_source_dir(DATASET, DATA_ROOT, fetched)
print('source folder on Drive:', SOURCE)

## 9. Copy the dataset to local disk

Training reads from `LOCAL_DATA` (much faster than Drive). Only missing or newer files are copied.
Run outputs still go to `OUT_DIR` on Drive.

In [ ]:
if SOURCE:
    sync(f'{DATA_ROOT}/social/{SOURCE}', f'{LOCAL_DATA}/social/{SOURCE}')
    RUN_DATA = LOCAL_DATA
else:
    print('[warn] no dataset folder on Drive yet: this session reads and writes data on Drive directly')
    RUN_DATA = DATA_ROOT
COMMON = f'out_dir={OUT_DIR} data.root={RUN_DATA}'
print(COMMON)

In [ ]:
!du -sh {RUN_DATA}/social/* 2>/dev/null || echo 'nothing local yet'

## 10. Build the subgraph cache (first session only) and save it to Drive

Loads the quickstart config's subgraphs once. The first time, this partitions the graph and writes
`social/<source>/processed/<hash>.pt`; later sessions find it and skip the work. New files are then copied
back to Drive so the next session reuses them.

In [ ]:
import gc, json
from gnn_mech.data import load_dataset

from gnn_mech.config import load_config
DATA_CFGS = [('quickstart', load_config(CONFIG, [f'out_dir={OUT_DIR}', f'data.root={RUN_DATA}']))]
seen = set()
for label, c in DATA_CFGS:
    d = c.to_dict()['data']
    d.pop('root', None)
    key = json.dumps(d, sort_keys=True, default=str)
    if key in seen:
        continue
    seen.add(key)
    try:
        tr, va, te, info = load_dataset(c.data)
        print(f'{label}: {len(tr)} / {len(va)} / {len(te)} subgraphs (train / val / test), '
              f'task {info.task}, feat_dim {getattr(info, "feat_dim", "?")}')
        del tr, va, te
    except Exception as e:
        print(f'[warn] could not prepare data for {label}: {e!r} (the run will try again)')
gc.collect()

if SOURCE is None:
    SOURCE = find_source_dir(DATASET, RUN_DATA)
if RUN_DATA != DATA_ROOT and SOURCE:
    sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
    print('cache saved to Drive:', f'{DATA_ROOT}/social/{SOURCE}')

## 11. Train one run (seed 0)

In [ ]:
!python -m gnn_mech.run --config {CONFIG} --seed 0 {COMMON}

## 12. Metrics (binary: `auroc`, `ap`; regression: `r2`, `mae`; multiclass: `acc`)

In [ ]:
import json, os
import numpy as np
import pandas as pd

PRIMARY = ('auroc', 'acc', 'r2', 'ap', 'mae', 'rmse')   # first one present is the headline metric
BASELINES = ('mlp', 'gcn')


def read_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None


def primary_metric(split):
    keys = [k for k, v in (split or {}).items() if k != 'loss' and isinstance(v, (int, float))]
    for k in PRIMARY:
        if k in keys:
            return k
    return keys[0] if keys else None

from gnn_mech.config import load_config

cfg = load_config(CONFIG, [f'out_dir={OUT_DIR}', f'data.root={RUN_DATA}', 'seed=0'])
RUN = Path(OUT_DIR) / cfg.exp_name / cfg.hash() / 'seed0'
print('run folder:', RUN)
metrics = read_json(RUN / 'metrics.json')
if metrics is None:
    print('no metrics.json yet: the run did not finish (rerun step 11)')
else:
    METRIC = primary_metric(metrics.get('test'))
    print(f"best epoch {metrics.get('best_epoch')} of {metrics.get('epochs_run')}, headline metric: {METRIC}")
    display(pd.DataFrame({s: metrics.get(s) or {} for s in ('val', 'test')}))

## 13. Training curves (train loss and validation metric per epoch)

In [ ]:
import matplotlib.pyplot as plt

hist = read_json(RUN / 'history.json') or []
if not hist:
    print('no history.json yet')
else:
    h = pd.DataFrame(hist)
    val_cols = [c for c in h.columns if c.startswith('val_') and c != 'val_loss']
    metric = f'val_{METRIC}' if metrics and f'val_{METRIC}' in h.columns else (val_cols[0] if val_cols else None)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    axes[0].plot(h['epoch'], h['train_loss'])
    axes[0].set(xlabel='epoch', ylabel='train loss', title='Train loss')
    if metric:
        axes[1].plot(h['epoch'], h[metric], color='C1')
        axes[1].set(xlabel='epoch', ylabel=metric, title='Validation ' + metric[4:])
    fig.tight_layout()
    plt.show()

### Save any new data files back to Drive

In [ ]:
if RUN_DATA != DATA_ROOT:
    SOURCE = SOURCE or find_source_dir(DATASET, RUN_DATA)
    if SOURCE:
        sync(f'{RUN_DATA}/social/{SOURCE}', f'{DATA_ROOT}/social/{SOURCE}')
        print('synced to Drive')